# Dark Matter Substructure Classification
### ConvNeXt V2 Large · ML4SCI DeepLense Task 1

**Goal:** Classify gravitational lens images into three dark matter substructure categories using a fine-tuned ConvNeXt V2 Large backbone.

| Class | Directory | Description |
|---|---|---|
| `no_sub` | `no/` | Smooth lens: no dark matter substructure |
| `subhalo` | `sphere/` | CDM subhalo: localised density perturbations |
| `vortex` | `vort/` | Vortex substructure: coherent angular momentum features |

**Evaluation Metric:** Macro One-vs-Rest ROC-AUC  
**Training:** 3-stage progressive fine-tuning · 90 total epochs (10 + 40 + 40)

## 1. Environment Setup
Install required packages.

In [30]:
!pip install timm albumentations -q

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import math, random, gc, warnings
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

import timm
from timm.utils import ModelEma

from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split

import albumentations as A
from albumentations.pytorch import ToTensorV2

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

print("✓ All imports successful")
print(f"  PyTorch  : {torch.__version__}")
print(f"  CUDA     : {torch.version.cuda}")
print(f"  timm     : {timm.__version__}")

✓ All imports successful
  PyTorch  : 2.9.0+cu126
  CUDA     : 12.6
  timm     : 1.0.24


## 2. Configuration

All hyperparameters are centralised in a single `CFG` class.  
Key design decisions:
- **Batch size 128**: 256 causes OOM when all 196M params + EMA + AdamW states are in VRAM simultaneously
- **Base LR 6.25e-4**: follows the linear scaling rule: `effective_lr = base_lr × batch / 256 = 3.125e-4`
- **Layer decay 0.7**: shallower layers receive progressively lower LR to preserve pretrained spatial features
- **BF16**: H100 native; no loss scaler needed unlike FP16

In [31]:
class CFG:
    # Paths 
    DATA_ROOT  = "/kaggle/input/datasets/stellarquant/deeplensetask1/dataset"
    OUTPUT_DIR = "/kaggle/working"

    # Model 
    MODEL_NAME      = "convnextv2_large.fcmae_ft_in22k_in1k_384"
    NUM_CLASSES     = 3
    IMG_SIZE        = 224
    DROP_PATH       = 0.1
    HEAD_INIT_SCALE = 0.001

    # Stage epochs 
    STAGE1_EPOCHS = 10   # frozen backbone, head-only warm-up
    STAGE2_EPOCHS = 40   # full fine-tuning with LLRD
    STAGE3_EPOCHS = 40   # low-LR cosine extension

    # DataLoader 
    BATCH_SIZE   = 128
    NUM_WORKERS  = 4
    VAL_SPLIT    = 0.10  # 90:10 stratified split from train/

    # Optimiser 
    BASE_LR      = 6.25e-4   # effective = BASE_LR × BATCH_SIZE / 256
    WEIGHT_DECAY = 0.05
    LAYER_DECAY  = 0.7
    MIN_LR       = 1e-6
    S3_LR        = 3.125e-5  # Stage 3: 10× lower than Stage 2 effective LR

    #Scheduler 
    S1_WARMUP = 2
    S2_WARMUP = 5

    #Regularisation 
    LABEL_SMOOTHING = 0.1
    MIXUP_ALPHA     = 0.4

    #EMA
    EMA_DECAY = 0.9999

    #Precision
    USE_BF16  = True
    GRAD_CLIP = 1.0

    #Dataset metadata
    CLASS_NAMES = ['no_sub', 'subhalo', 'vortex']
    CLASS_DIRS  = {'no_sub': 'no', 'subhalo': 'sphere', 'vortex': 'vort'}
    PIXEL_MEAN  = 0.0615
    PIXEL_STD   = 0.1152
    SEED        = 42

os.makedirs(CFG.OUTPUT_DIR, exist_ok=True)
print("✓ Config loaded")

✓ Config loaded


## 3. Reproducibility
Seed all random number generators to ensure fully reproducible training runs.

In [32]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

seed_everything(CFG.SEED)
print(f"Seed set to {CFG.SEED}")

Seed set to 42


## 4. Dataset

The `LensDataset` class loads `.npy` gravitational lens images.  

**Pipeline per image:**
1. Load `(1, 150, 150)` float64 array → squeeze to `(150, 150)` float32
2. Scale to uint8 `[0, 255]` for Albumentations compatibility
3. Apply augmentation transforms → normalised `(1, H, W)` float32 tensor

**Auto-discovery** walks `/kaggle/input` to find the dataset root if the hardcoded path doesn't exist, handling slug variations automatically.

In [33]:
class LensDataset(Dataset):
    def __init__(self, file_paths, labels, transform=None):
        self.file_paths = file_paths
        self.labels     = labels
        self.transform  = transform

    def __len__(self): return len(self.file_paths)

    def __getitem__(self, idx):
        img = np.load(self.file_paths[idx]).astype(np.float32)
        if img.ndim == 3: img = img[0]          # (1,150,150) → (150,150)
        img_u8 = (img * 255).clip(0, 255).astype(np.uint8)
        if self.transform:
            return self.transform(image=img_u8)['image'], self.labels[idx]
        return torch.from_numpy(img_u8[None]).float() / 255.0, self.labels[idx]


def auto_discover_root(base='/kaggle/input') -> Path:
    expected = set(CFG.CLASS_DIRS.values())
    for dirpath, dirnames, _ in os.walk(base):
        p = Path(dirpath)
        if {'train', 'val'}.issubset(set(dirnames)):
            train_p = p / 'train'
            if train_p.exists():
                found = {d.name for d in train_p.iterdir() if d.is_dir()}
                if expected & found:
                    print(f"Auto-discovered DATA_ROOT = {p}")
                    return p
    raise FileNotFoundError(
        "Could not auto-discover dataset. "
        "Run: [print(r) for r,d,f in os.walk('/kaggle/input')]"
    )


def build_file_list(root: Path, split: str):
    paths, labels = [], []
    split_dir = root / split
    if not split_dir.exists():
        raise FileNotFoundError(
            f"Split dir missing: {split_dir}\n"
            f"Root contents: {list(root.iterdir()) if root.exists() else 'ROOT MISSING'}"
        )
    for i, cls in enumerate(CFG.CLASS_NAMES):
        cls_dir   = split_dir / CFG.CLASS_DIRS[cls]
        if not cls_dir.exists():
            raise FileNotFoundError(
                f"Class dir missing: {cls_dir}\n"
                f"Split contents: {list(split_dir.iterdir())}"
            )
        npy_files = sorted(cls_dir.glob("*.npy")) + sorted(cls_dir.glob("*.NPY"))
        if not npy_files:
            raise FileNotFoundError(
                f"No .npy files in {cls_dir}\n"
                f"Found: {list(cls_dir.iterdir())[:8]}"
            )
        paths.extend(str(p) for p in npy_files)
        labels.extend([i] * len(npy_files))
        print(f"  [{split}/{cls}]  {len(npy_files):,} images  ← {cls_dir}")
    return paths, labels

print("Dataset classes defined")

Dataset classes defined


## 5. Data Loading

Build file lists, apply the 90:10 stratified train/val split, and instantiate DataLoaders.  
The `val/` folder (7,500 images) is the **fixed held-out test set**, never used for any tuning decision.

In [34]:
data_root = Path(CFG.DATA_ROOT)
if not data_root.exists():
    print("Configured path not found, running auto-discovery...")
    data_root = auto_discover_root()
else:
    print(f"DATA_ROOT = {data_root}")

print("\n Train split")
train_paths, train_labels = build_file_list(data_root, 'train')
print("\n Test split")
test_paths,  test_labels  = build_file_list(data_root, 'val')

tr_paths, val_paths, tr_labels, val_labels = train_test_split(
    train_paths, train_labels,
    test_size=CFG.VAL_SPLIT, stratify=train_labels,
    random_state=CFG.SEED,
)

print(f"\n  Train : {len(tr_paths):,}  |  Val : {len(val_paths):,}  "
      f"|  Test : {len(test_paths):,}")
print(f"  Class balance (train): "
      + str({c: tr_labels.count(i) for i, c in enumerate(CFG.CLASS_NAMES)}))

DATA_ROOT = /kaggle/input/datasets/stellarquant/deeplensetask1/dataset

 Train split
  [train/no_sub]  10,000 images  ← /kaggle/input/datasets/stellarquant/deeplensetask1/dataset/train/no
  [train/subhalo]  10,000 images  ← /kaggle/input/datasets/stellarquant/deeplensetask1/dataset/train/sphere
  [train/vortex]  10,000 images  ← /kaggle/input/datasets/stellarquant/deeplensetask1/dataset/train/vort

 Test split
  [val/no_sub]  2,500 images  ← /kaggle/input/datasets/stellarquant/deeplensetask1/dataset/val/no
  [val/subhalo]  2,500 images  ← /kaggle/input/datasets/stellarquant/deeplensetask1/dataset/val/sphere
  [val/vortex]  2,500 images  ← /kaggle/input/datasets/stellarquant/deeplensetask1/dataset/val/vort

  Train : 27,000  |  Val : 3,000  |  Test : 7,500
  Class balance (train): {'no_sub': 9000, 'subhalo': 9000, 'vortex': 9000}


## 6. Augmentation Pipeline

**Why each augmentation was chosen:**

| Transform | Reason |
|---|---|
| `Resize(224×224, BICUBIC)` | Match model pretraining resolution |
| `HorizontalFlip / VerticalFlip` | Label-preserving; lens looks same flipped |
| `Rotate(limit=180, p=0.9)` | **Critical**: gravitational lenses are rotationally symmetric; full 360° coverage |
| `RandomResizedCrop(scale=0.90–1.00)` | Conservative ±10%, wider scale would crop out the Einstein ring |
| `GaussNoise(σ≈0.01–0.02)` | Simulates telescope CCD read noise |
| `CoarseDropout` | Simulates foreground star/galaxy occlusion |
| ❌ No colour jitter | Intensity is pre-normalised; all 3 classes have identical pixel distributions — jitter would destroy the subtle spatial signal |

In [35]:
def get_train_transforms():
    return A.Compose([
        A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE, interpolation=2),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.Rotate(limit=180, p=0.9, border_mode=0, value=0),
        A.RandomResizedCrop(
            size=(CFG.IMG_SIZE, CFG.IMG_SIZE),
            scale=(0.90, 1.00), ratio=(0.95, 1.05),
            interpolation=2, p=0.5,
        ),
        A.GaussNoise(var_limit=(0.65, 2.60), p=0.35),
        A.CoarseDropout(
            max_holes=4, max_height=18, max_width=18,
            min_holes=1, min_height=8,  min_width=8,
            fill_value=0, p=0.20,
        ),
        A.Normalize(mean=[CFG.PIXEL_MEAN], std=[CFG.PIXEL_STD],
                    max_pixel_value=255.0),
        ToTensorV2(),
    ])


def get_val_transforms():
    return A.Compose([
        A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE, interpolation=2),
        A.Normalize(mean=[CFG.PIXEL_MEAN], std=[CFG.PIXEL_STD],
                    max_pixel_value=255.0),
        ToTensorV2(),
    ])

print("Augmentation pipelines defined")

Augmentation pipelines defined


In [36]:
train_loader = DataLoader(
    LensDataset(tr_paths,   tr_labels,   get_train_transforms()),
    batch_size=CFG.BATCH_SIZE, shuffle=True,
    num_workers=CFG.NUM_WORKERS, pin_memory=True,
    drop_last=True, persistent_workers=True,
)
val_loader = DataLoader(
    LensDataset(val_paths,  val_labels,  get_val_transforms()),
    batch_size=CFG.BATCH_SIZE * 2, shuffle=False,
    num_workers=CFG.NUM_WORKERS, pin_memory=True, persistent_workers=True,
)
test_loader = DataLoader(
    LensDataset(test_paths, test_labels, get_val_transforms()),
    batch_size=CFG.BATCH_SIZE * 2, shuffle=False,
    num_workers=CFG.NUM_WORKERS, pin_memory=True, persistent_workers=True,
)

print(f"DataLoaders ready")
print(f"Train batches : {len(train_loader):,}  "
      f"(batch={CFG.BATCH_SIZE}, drop_last=True)")
print(f"Val   batches : {len(val_loader):,}  "
      f"(batch={CFG.BATCH_SIZE*2})")
print(f"Test  batches : {len(test_loader):,}  "
      f"(batch={CFG.BATCH_SIZE*2})")

DataLoaders ready
Train batches : 210  (batch=128, drop_last=True)
Val   batches : 12  (batch=256)
Test  batches : 30  (batch=256)


## 7. Model — ConvNeXt V2 Large

**Architecture:** `convnextv2_large.fcmae_ft_in22k_in1k_384`  
**Params:** 196.4M

**Key configuration choices:**

| Choice | Value | Reason |
|---|---|---|
| `in_chans=3` | Grayscale replicated 3× on GPU | Preserves all 196M pretrained FCMAE weights |
| `head_init_scale=0.001` | Near-zero linear head init | Prevents large random head gradients from corrupting backbone in early training |
| Gradient checkpointing | `enable=True` | Recomputes activations during backward; saves ~35% VRAM at ~20% compute cost, essential for H100 80GB with full unfreeze + EMA + AdamW |
| `drop_path_rate=0.1` | Stochastic depth | Randomly drops residual branches; regularises the deep 27-block Stage 2 |

**Channel replication** happens in the training loop on GPU:
```python
x = x.repeat(1, 3, 1, 1) 
```

In [37]:
def replicate_channels(x):
    return x.repeat(1, 3, 1, 1)   # (B,1,H,W) → (B,3,H,W)


def build_model() -> nn.Module:
    model = timm.create_model(
        CFG.MODEL_NAME, pretrained=True,
        num_classes=CFG.NUM_CLASSES,
        drop_path_rate=CFG.DROP_PATH,
        in_chans=3,
    )
    # Near-zero head init, prevents early gradient explosion from random head
    head = getattr(model, 'head', None)
    if head is not None:
        fc = getattr(head, 'fc', head if isinstance(head, nn.Linear) else None)
        if isinstance(fc, nn.Linear):
            nn.init.trunc_normal_(fc.weight, std=0.02 * CFG.HEAD_INIT_SCALE)
            nn.init.constant_(fc.bias, 0)
    # Gradient checkpointing, trades compute for memory
    model.set_grad_checkpointing(enable=True)
    total = sum(p.numel() for p in model.parameters())
    print(f"Model : {CFG.MODEL_NAME}")
    print(f"Params: {total/1e6:.1f}M  |  grad_ckpt=ON")
    return model

print("Model builder defined")

Model builder defined


## 8. Layer-wise Learning Rate Decay (LLRD)

LLRD assigns progressively lower learning rates to shallower layers, protecting pretrained spatial features from being overwritten by large gradients from the randomly-initialised head.
```
Head        → base_lr × 1.0      (full rate)
stages.3    → base_lr × 0.7¹
stages.2    → base_lr × 0.7²
stages.1    → base_lr × 0.7³
stages.0    → base_lr × 0.7⁴
stem        → base_lr × 0.7⁵    (most protected)
remaining   → base_lr × 0.7⁶    (fallback)
```

Each prefix is split into **two sub-groups**: one with weight decay (0.05) for weights, and one without for bias/norm/GRN terms, giving **12 total param groups** as shown in the training log.

In [38]:
def get_llrd_param_groups(model: nn.Module, base_lr: float) -> list:
    d     = CFG.LAYER_DECAY
    no_wd = {'bias','norm','bn','ln','gamma','beta','LayerNorm','grn','scale'}

    layer_map = {
        'head':     1.0,
        'norm_pre': d ** 1,
        'stages.3': d ** 1,
        'stages.2': d ** 2,
        'stages.1': d ** 3,
        'stages.0': d ** 4,
        'stem':     d ** 5,
    }

    groups, assigned = [], set()

    def skip_wd(name): return any(k in name for k in no_wd)

    for prefix, scale in layer_map.items():
        wd_p, no_wd_p = [], []
        for name, param in model.named_parameters():
            if not param.requires_grad or name in assigned: continue
            if not name.startswith(prefix): continue
            assigned.add(name)
            (no_wd_p if skip_wd(name) else wd_p).append(param)
        if wd_p:
            groups.append({'params': wd_p,    'lr': base_lr * scale,
                           'weight_decay': CFG.WEIGHT_DECAY})
        if no_wd_p:
            groups.append({'params': no_wd_p, 'lr': base_lr * scale,
                           'weight_decay': 0.0})

    remaining = [(n, p) for n, p in model.named_parameters()
                 if p.requires_grad and n not in assigned]
    if remaining:
        groups.append({'params': [p for _, p in remaining],
                       'lr': base_lr * d**6, 'weight_decay': CFG.WEIGHT_DECAY})

    total_p = sum(p.numel() for g in groups for p in g['params'])
    print(f"  LLRD  : {len(groups)} groups | base_lr={base_lr:.2e} "
          f"| decay={d} | {total_p/1e6:.1f}M params")
    return groups

print("LLRD param group builder defined")

LLRD param group builder defined


## 9. Scheduler, Mixup & Loss

**Cosine decay with linear warmup**: standard for fine-tuning large pretrained ConvNets. The warmup prevents large LR updates before the optimiser accumulates reliable gradient statistics.

**Mixup (α=0.4)** blends pairs of training images and their labels:  
`loss = λ·CE(pred, y_a) + (1−λ)·CE(pred, y_b)`  

CutMix is intentionally excluded,it excises rectangular patches from images, which can literally remove the subhalo signal (a localised perturbation occupying a small spatial region). Mixup blends globally and preserves all spatial signal at reduced amplitude.

**Label smoothing (0.1)** softens hard one-hot targets, preventing overconfident predictions on the physically subtle `subhalo`/`no_sub` boundary.

In [39]:
def cosine_with_warmup(optimizer, warmup_epochs: int, total_epochs: int,
                        min_lr_ratio: float = 0.01):
    def _fn(ep):
        if ep < warmup_epochs:
            return max(1e-6, (ep + 1) / warmup_epochs)
        prog = (ep - warmup_epochs) / max(1, total_epochs - warmup_epochs)
        return min_lr_ratio + 0.5 * (1.0 - min_lr_ratio) * (1 + math.cos(math.pi * prog))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, _fn)


def mixup_data(x, y, alpha=0.4):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam


def mixup_loss(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

print("Scheduler and Mixup defined")

Scheduler and Mixup defined


## 10. Evaluation: Macro OvR ROC-AUC

The primary metric is **Macro One-vs-Rest AUC**:
```python
roc_auc_score(y_true, y_prob, multi_class='ovr', average='macro')
```
Per-class AUC is also tracked to diagnose which class boundary is hardest.  
All evaluations run on the **EMA model** weights, not the live training weights.  
Softmax probabilities (not hard predictions) are passed to `roc_auc_score`.

In [40]:
@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    dtype = torch.bfloat16 if CFG.USE_BF16 else torch.float32
    all_probs, all_labels = [], []
    for imgs, labels in tqdm(loader, desc='Eval', leave=False):
        imgs = replicate_channels(imgs).to(device, non_blocking=True)
        with torch.autocast('cuda', dtype=dtype):
            probs = F.softmax(model(imgs), dim=-1)
        all_probs.append(probs.cpu().float().numpy())
        all_labels.append(labels.numpy())
    probs  = np.concatenate(all_probs)
    labels = np.concatenate(all_labels)
    macro  = roc_auc_score(labels, probs, multi_class='ovr', average='macro')
    per_cls = {cls: roc_auc_score((labels==i).astype(int), probs[:,i])
               for i, cls in enumerate(CFG.CLASS_NAMES)}
    return macro, per_cls, probs, labels

print("Evaluate function defined")

Evaluate function defined


## 11. Training Loop

**BF16 mixed precision** via `torch.autocast('cuda', dtype=torch.bfloat16)`:
- Same dynamic range as FP32 (8 exponent bits), no loss scaler needed
- H100 native tensor core acceleration, ~2× throughput vs FP32

**Gradient clipping** (`max_norm=1.0`) prevents occasional gradient explosions during the first few unfrozen epochs.

**EMA update** happens every step after the optimizer step. The EMA model accumulates a running average of all weight snapshots, making it more robust to late-training noise.

In [41]:
def train_one_epoch(model, loader, optimizer, criterion, scheduler,
                    ema, device, label: str) -> float:
    model.train()
    dtype  = torch.bfloat16 if CFG.USE_BF16 else torch.float32
    losses = []
    pbar   = tqdm(loader, desc=label, leave=True)
    for imgs, labels in pbar:
        imgs   = replicate_channels(imgs).to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        imgs, y_a, y_b, lam = mixup_data(imgs, labels, CFG.MIXUP_ALPHA)
        with torch.autocast('cuda', dtype=dtype):
            logits = model(imgs)
            loss   = mixup_loss(criterion, logits, y_a, y_b, lam)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), CFG.GRAD_CLIP)
        optimizer.step()
        ema.update(model)
        losses.append(loss.item())
        pbar.set_postfix({'loss': f'{loss.item():.4f}',
                          'lr':   f'{optimizer.param_groups[0]["lr"]:.2e}'})
    scheduler.step()
    return float(np.mean(losses))

print("Training loop defined")

Training loop defined


## 12. Plotting Utilities

In [42]:
def plot_roc_curves(labels, probs, save_path, title="Test Set"):
    COLORS = ['#e74c3c', '#2ecc71', '#3498db']
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle(f'ROC Curves — {title}', fontsize=14, fontweight='bold')
    ax, per_aucs = axes[0], []
    for i, (cls, col) in enumerate(zip(CFG.CLASS_NAMES, COLORS)):
        binary      = (labels == i).astype(int)
        fpr, tpr, _ = roc_curve(binary, probs[:, i])
        auc_val     = roc_auc_score(binary, probs[:, i])
        per_aucs.append(auc_val)
        ax.plot(fpr, tpr, color=col, lw=2.2, label=f'{cls}  (AUC = {auc_val:.4f})')
    ax.plot([0,1],[0,1],'k--',lw=1,label='Random')
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
    ax.set_title('One-vs-Rest ROC per Class')
    ax.legend(loc='lower right'); ax.grid(alpha=0.25)
    ax.set_xlim([0,1]); ax.set_ylim([0,1.02])
    macro = np.mean(per_aucs)
    ax2   = axes[1]
    bars  = ax2.bar(CFG.CLASS_NAMES, per_aucs, color=COLORS,
                    alpha=0.85, edgecolor='white', linewidth=1.2)
    ax2.axhline(macro, color='gold', lw=2, ls='--', label=f'Macro = {macro:.4f}')
    ax2.set_ylim(max(0.5, min(per_aucs)-0.03), 1.005)
    ax2.set_ylabel('AUC'); ax2.set_title('Per-Class AUC Summary')
    ax2.legend(); ax2.grid(axis='y', alpha=0.25)
    for bar, val in zip(bars, per_aucs):
        ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.001,
                 f'{val:.4f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  ROC plot  → {save_path}")


def plot_history(history, save_path):
    total = len(history['train_loss'])
    eps   = range(1, total + 1)
    fig, ax1 = plt.subplots(figsize=(13, 5))
    ax1.plot(eps, history['train_loss'], color='#e74c3c', lw=2, label='Train Loss')
    s1 = CFG.STAGE1_EPOCHS
    s2 = s1 + CFG.STAGE2_EPOCHS
    ax1.axvline(s1 + 0.5, color='gray',   ls=':', lw=1.5, label='S1→S2')
    ax1.axvline(s2 + 0.5, color='orange', ls=':', lw=1.5, label='S2→S3')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss', color='#e74c3c')
    ax2 = ax1.twinx()
    ax2.plot(eps, history['val_auc'], color='#2ecc71', lw=2,
             label='Val Macro AUC (EMA)')
    ax2.set_ylabel('AUC', color='#2ecc71')
    lines  = ax1.get_legend_handles_labels()
    lines2 = ax2.get_legend_handles_labels()
    ax1.legend(lines[0]+lines2[0], lines[1]+lines2[1], loc='center right')
    ax1.set_title('Training History — ConvNeXt V2 Large  (S1 + S2 + S3)')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  History   → {save_path}")

print("Plotting utilities defined")

Plotting utilities defined


## 13.Model Initialisation

In [43]:
if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA not available.\n"
        "In Kaggle: Settings → Accelerator → GPU (H100), then restart kernel."
    )
device = torch.device('cuda')

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU  : {gpu_name}  ({vram_gb:.1f} GB)")
print(f"BF16 : {CFG.USE_BF16}  |  Batch : {CFG.BATCH_SIZE}  |  "
      f"Epochs : S1={CFG.STAGE1_EPOCHS} + S2={CFG.STAGE2_EPOCHS} "
      f"+ S3={CFG.STAGE3_EPOCHS} = "
      f"{CFG.STAGE1_EPOCHS+CFG.STAGE2_EPOCHS+CFG.STAGE3_EPOCHS} total")

model     = build_model().to(device)
ema       = ModelEma(model, decay=CFG.EMA_DECAY, device=device)
criterion = nn.CrossEntropyLoss(label_smoothing=CFG.LABEL_SMOOTHING)
ckpt_path = f"{CFG.OUTPUT_DIR}/best_model.pth"

history      = {'train_loss': [], 'val_auc': []}
best_val_auc = 0.0

def _log(stage_label, ep, total, loss, macro, per_cls):
    cls_str = '  '.join(f'{k}={v:.4f}' for k, v in per_cls.items())
    print(f"  [{stage_label} {ep:02d}/{total}]  "
          f"loss={loss:.4f}  macro_auc={macro:.4f}  |  {cls_str}")

def _maybe_save(macro, ep_abs):
    global best_val_auc
    if macro > best_val_auc:
        best_val_auc = macro
        torch.save({'epoch': ep_abs, 'model': ema.ema.state_dict(),
                    'val_auc': macro}, ckpt_path)
        print(f"  ✓  New best val AUC={macro:.4f}  → checkpoint saved")

GPU  : NVIDIA H100 80GB HBM3  (85.0 GB)
BF16 : True  |  Batch : 128  |  Epochs : S1=10 + S2=40 + S3=40 = 90 total
Model : convnextv2_large.fcmae_ft_in22k_in1k_384
Params: 196.4M  |  grad_ckpt=ON


## 14. Stage 1: Head-Only Warm-up

**Rationale:** The randomly-initialised classification head produces large, noisy gradients in the first few steps. If the backbone is already unfrozen at this point, those gradients propagate backwards through all 196M parameters and corrupt the pretrained FCMAE spatial features.

**Strategy:** Freeze the entire backbone. Train only the 3-class linear head for 10 epochs at LR=1e-3 with a 2-epoch linear warmup and cosine decay. By epoch 10 the head has adapted to the 3-class lens task, and subsequent backbone unfreezing starts from a sensible gradient landscape.

In [44]:
print(f"\n{'═'*62}")
print(f"  STAGE 1: Head-only ({CFG.STAGE1_EPOCHS} epochs, backbone frozen)")
print(f"{'═'*62}\n")

for name, p in model.named_parameters():
    p.requires_grad = ('head' in name)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  Trainable params: {trainable:,}  (head only)")

s1_opt   = AdamW([p for p in model.parameters() if p.requires_grad],
                 lr=1e-3, weight_decay=CFG.WEIGHT_DECAY)
s1_sched = cosine_with_warmup(s1_opt, CFG.S1_WARMUP, CFG.STAGE1_EPOCHS)

for ep in range(CFG.STAGE1_EPOCHS):
    loss = train_one_epoch(model, train_loader, s1_opt, criterion,
                           s1_sched, ema, device,
                           f'S1 {ep+1:02d}/{CFG.STAGE1_EPOCHS}')
    macro, per_cls, _, _ = evaluate(ema.ema, val_loader, device)
    history['train_loss'].append(loss)
    history['val_auc'].append(macro)
    _log('S1', ep+1, CFG.STAGE1_EPOCHS, loss, macro, per_cls)
    _maybe_save(macro, ep)


══════════════════════════════════════════════════════════════
  STAGE 1: Head-only (10 epochs, backbone frozen)
══════════════════════════════════════════════════════════════

  Trainable params: 7,683  (head only)


S1 01/10:   0%|          | 0/210 [00:00<?, ?it/s]

Eval:   0%|          | 0/12 [00:00<?, ?it/s]

  [S1 01/10]  loss=1.1004  macro_auc=0.6237  |  no_sub=0.6599  subhalo=0.6322  vortex=0.5790
  ✓  New best val AUC=0.6237  → checkpoint saved


S1 02/10:   0%|          | 0/210 [00:00<?, ?it/s]

...

Eval:   0%|          | 0/12 [00:00<?, ?it/s]

  [S1 09/10]  loss=1.0872  macro_auc=0.6801  |  no_sub=0.7368  subhalo=0.6757  vortex=0.6278
  ✓  New best val AUC=0.6801  → checkpoint saved


S1 10/10:   0%|          | 0/210 [00:00<?, ?it/s]

Eval:   0%|          | 0/12 [00:00<?, ?it/s]

  [S1 10/10]  loss=1.0850  macro_auc=0.6816  |  no_sub=0.7389  subhalo=0.6767  vortex=0.6292
  ✓  New best val AUC=0.6816  → checkpoint saved


## 15. Stage 2: Full Fine-tuning with LLRD

**Memory management:** Before unfreezing all 196M parameters, the Stage-1 optimizer is explicitly deleted and CUDA cache is flushed. AdamW holds 2 momentum buffers per parameter — deleting the S1 optimizer frees ~1.5 GB before the much larger S2 optimizer is created.

**LLRD:** Shallower layers receive lower learning rates (stem gets `base_lr × 0.7⁵`), preserving low-level spatial features like edge detectors and ring structure encoders that transfer well from ImageNet pretraining.

**Linear scaling rule:** `effective_lr = 6.25e-4 × 128/256 = 3.125e-4`

In [45]:
print(f"\n{'═'*62}")
print(f"  STAGE 2: Full LLRD fine-tuning ({CFG.STAGE2_EPOCHS} epochs)")
print(f"{'═'*62}\n")

del s1_opt, s1_sched
gc.collect(); torch.cuda.empty_cache()
print(f"  GPU after cache flush: "
      f"{torch.cuda.memory_allocated()/1e9:.1f} GB alloc / "
      f"{torch.cuda.memory_reserved()/1e9:.1f} GB reserved\n")

for p in model.parameters(): p.requires_grad = True

eff_lr    = CFG.BASE_LR * CFG.BATCH_SIZE / 256
s2_groups = get_llrd_param_groups(model, eff_lr)
s2_opt    = AdamW(s2_groups, weight_decay=CFG.WEIGHT_DECAY)
s2_sched  = cosine_with_warmup(
    s2_opt, CFG.S2_WARMUP, CFG.STAGE2_EPOCHS,
    min_lr_ratio=CFG.MIN_LR / eff_lr,
)

for ep in range(CFG.STAGE2_EPOCHS):
    abs_ep = CFG.STAGE1_EPOCHS + ep
    loss   = train_one_epoch(model, train_loader, s2_opt, criterion,
                             s2_sched, ema, device,
                             f'S2 {ep+1:02d}/{CFG.STAGE2_EPOCHS}')
    macro, per_cls, _, _ = evaluate(ema.ema, val_loader, device)
    history['train_loss'].append(loss)
    history['val_auc'].append(macro)
    _log('S2', ep+1, CFG.STAGE2_EPOCHS, loss, macro, per_cls)
    _maybe_save(macro, abs_ep)


══════════════════════════════════════════════════════════════
  STAGE 2: Full LLRD fine-tuning (40 epochs)
══════════════════════════════════════════════════════════════

  GPU after cache flush: 1.6 GB alloc / 1.7 GB reserved

  LLRD  : 12 groups | base_lr=3.13e-04 | decay=0.7 | 196.4M params


S2 01/40:   0%|          | 0/210 [00:00<?, ?it/s]

Eval:   0%|          | 0/12 [00:00<?, ?it/s]

  [S2 01/40]  loss=1.0352  macro_auc=0.6872  |  no_sub=0.7449  subhalo=0.6835  vortex=0.6332
  ✓  New best val AUC=0.6872  → checkpoint saved


S2 02/40:   0%|          | 0/210 [00:00<?, ?it/s]

...

Eval:   0%|          | 0/12 [00:00<?, ?it/s]

  [S2 39/40]  loss=0.7733  macro_auc=0.9734  |  no_sub=0.9833  subhalo=0.9676  vortex=0.9693
  ✓  New best val AUC=0.9734  → checkpoint saved


S2 40/40:   0%|          | 0/210 [00:00<?, ?it/s]

Eval:   0%|          | 0/12 [00:00<?, ?it/s]

  [S2 40/40]  loss=0.7479  macro_auc=0.9748  |  no_sub=0.9840  subhalo=0.9692  vortex=0.9713
  ✓  New best val AUC=0.9748  → checkpoint saved


## 16. Stage 3: Low-LR Extension

The model is well-converged after Stage 2. A cosine cooldown at 10× lower LR (`3.125e-5`) with no warmup refines subtle decision boundaries, particularly the `subhalo`/`no_sub` boundary where pixel-level statistics are nearly identical and the model must distinguish purely by spatial pattern density.

The S2 optimizer is deleted before S3 to free memory before rebuilding LLRD groups at the new lower base LR.

In [46]:
print(f"\n{'═'*62}")
print(f"  STAGE 3: Extension ({CFG.STAGE3_EPOCHS} epochs, LR={CFG.S3_LR:.2e})")
print(f"{'═'*62}\n")

del s2_opt, s2_sched
gc.collect(); torch.cuda.empty_cache()

s3_groups = get_llrd_param_groups(model, CFG.S3_LR)
s3_opt    = AdamW(s3_groups, weight_decay=CFG.WEIGHT_DECAY)
s3_sched  = cosine_with_warmup(
    s3_opt, warmup_epochs=0, total_epochs=CFG.STAGE3_EPOCHS,
    min_lr_ratio=0.01,
)

for ep in range(CFG.STAGE3_EPOCHS):
    abs_ep = CFG.STAGE1_EPOCHS + CFG.STAGE2_EPOCHS + ep
    loss   = train_one_epoch(model, train_loader, s3_opt, criterion,
                             s3_sched, ema, device,
                             f'S3 {ep+1:02d}/{CFG.STAGE3_EPOCHS}')
    macro, per_cls, _, _ = evaluate(ema.ema, val_loader, device)
    history['train_loss'].append(loss)
    history['val_auc'].append(macro)
    _log('S3', ep+1, CFG.STAGE3_EPOCHS, loss, macro, per_cls)
    _maybe_save(macro, abs_ep)


══════════════════════════════════════════════════════════════
  STAGE 3: Extension (40 epochs, LR=3.13e-05)
══════════════════════════════════════════════════════════════

  LLRD  : 12 groups | base_lr=3.13e-05 | decay=0.7 | 196.4M params


S3 01/40:   0%|          | 0/210 [00:00<?, ?it/s]

Eval:   0%|          | 0/12 [00:00<?, ?it/s]

  [S3 01/40]  loss=0.7772  macro_auc=0.9762  |  no_sub=0.9846  subhalo=0.9706  vortex=0.9735
  ✓  New best val AUC=0.9762  → checkpoint saved


S3 02/40:   0%|          | 0/210 [00:00<?, ?it/s]

...

Eval:   0%|          | 0/12 [00:00<?, ?it/s]

  [S3 39/40]  loss=0.7531  macro_auc=0.9936  |  no_sub=0.9933  subhalo=0.9909  vortex=0.9965
  ✓  New best val AUC=0.9936  → checkpoint saved


S3 40/40:   0%|          | 0/210 [00:00<?, ?it/s]

Eval:   0%|          | 0/12 [00:00<?, ?it/s]

  [S3 40/40]  loss=0.7330  macro_auc=0.9936  |  no_sub=0.9933  subhalo=0.9912  vortex=0.9964
  ✓  New best val AUC=0.9936  → checkpoint saved


## 17. Final Test Evaluation & Results

Load the best EMA checkpoint (saved whenever val AUC improved across all 90 epochs) and evaluate on the held-out test set (7,500 images, never seen during training or validation).

In [47]:
print(f"\n{'═'*62}")
print("  FINAL TEST EVALUATION (best EMA checkpoint)")
print(f"{'═'*62}\n")

ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
model.load_state_dict(ckpt['model'])

test_auc, test_per_cls, test_probs, test_lbl = evaluate(model, test_loader, device)

print(f"  Macro OvR AUC : {test_auc:.4f}")
for cls, val in test_per_cls.items():
    print(f"    {cls:>10} : {val:.4f}")

plot_roc_curves(
    test_lbl, test_probs,
    save_path=f"{CFG.OUTPUT_DIR}/roc_curves_final.png",
    title=f"Test Set — ConvNeXt V2 Large  (Macro AUC = {test_auc:.4f})",
)
plot_history(history, save_path=f"{CFG.OUTPUT_DIR}/training_history.png")

print(f"\n{'─'*62}")
print(f"  Best val AUC  : {best_val_auc:.4f}")
print(f"  Test AUC      : {test_auc:.4f}")
print(f"  Checkpoint    : {ckpt_path}")
print(f"  Outputs       : {CFG.OUTPUT_DIR}")
print(f"{'─'*62}")


══════════════════════════════════════════════════════════════
  FINAL TEST EVALUATION (best EMA checkpoint)
══════════════════════════════════════════════════════════════



Eval:   0%|          | 0/30 [00:00<?, ?it/s]

  Macro OvR AUC : 0.9934
        no_sub : 0.9929
       subhalo : 0.9909
        vortex : 0.9963
  ROC plot  → /kaggle/working/roc_curves_final.png
  History   → /kaggle/working/training_history.png

──────────────────────────────────────────────────────────────
  Best val AUC  : 0.9936
  Test AUC      : 0.9934
  Checkpoint    : /kaggle/working/best_model.pth
  Outputs       : /kaggle/working
──────────────────────────────────────────────────────────────
